# 레슨 17 - Foundry Local과 Qwen을 사용한 로컬 AI 에이전트 생성

이 노트북에서는 워크스테이션에서 완전히 실행되는 <strong>로컬 엔지니어링 어시스턴트</strong>를 만듭니다. 클라우드 추론은 전혀 사용되지 않습니다. 어시스턴트는 다음을 수행할 수 있습니다:

1. Foundry Local을 통해 Qwen 함수 호출로 **도구 호출**.
2. 샌드박스 프로젝트 디렉터리 내의 **파일 나열 및 읽기**.
3. 간단한 메트릭으로 **코드 분석**.
4. 로컬 RAG(Chroma)로 **문서 검색**.
5. 로컬 MCP 서버를 <strong>사용</strong> (구성되지 않은 경우 자동으로 건너뜀).

에이전트 코드는 클라우드 레슨과 거의 동일하며 — 클라이언트 엔드포인트만 클라우드에서 `localhost`로 이동합니다.


## 설정

이 노트북을 실행하기 전에:

1. **Microsoft Foundry Local 설치** (운영 체제에 맞는 [문서](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) 참조).
2. **Qwen 모델 다운로드 및 시작:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. 아래 Python 패키지 설치.

모든 작업은 로컬에서 실행됩니다. 약 8GB RAM을 가진 기계가 현실적인 최소 사양입니다.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Foundry 로컬에 연결하기

`FoundryLocalManager`는 필요 시 모델을 다운로드하고, 로컬 서비스를 시작하며, 우리에게 <strong>OpenAI 호환 엔드포인트</strong>를 제공합니다. 그런 다음 표준 OpenAI SDK를 해당 엔드포인트로 지정합니다. API 키는 로컬 플레이스홀더이며 — 클라우드 자격 증명은 포함되지 않습니다.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. 로컬 도구 (샌드박스 파일 작업)

우리는 디스크에 작은 샘플 프로젝트를 만든 후, 해당 프로젝트 루트에 범위가 지정된 도구를 정의합니다. 샌드박스 검사는 로컬에서도 중요합니다: 임의 경로를 읽는 도구는 사용자의 권한으로 실행되며 사용자가 접근할 수 있는 모든 것에 접근할 수 있습니다.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Chroma를 이용한 로컬 RAG

우리는 소규모 문서 스니펫 집합을 로컬 Chroma 컬렉션에 임베딩합니다. Chroma는 프로세스 내에서 실행되며 벡터를 디스크에 저장합니다 — 서버나 클라우드가 필요 없습니다. `search_docs` 도구는 쿼리에 가장 관련성이 높은 스니펫을 검색합니다.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. 도구 호출 루프

이제 OpenAI 도구 스키마를 사용하여 도구를 모델에 등록하고 표준 도구 호출 루프를 실행합니다 — 모델이 도구를 요청하면, 이를 로컬에서 실행하고 결과를 다시 제공하며, 모델이 최종 답변을 생성할 때까지 반복합니다. Qwen의 신뢰할 수 있는 함수 호출이 이 기능을 온디바이스에서 가능하게 합니다.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. 로컬 MCP (선택 사항)

MCP는 클라우드 서비스가 아닌 전송 수단입니다 — MCP 서버는 `stdio`를 통해 로컬 프로세스로 실행할 수 있습니다. 아래 셀은 로컬 MCP 서버(예: 파일 시스템 서버)가 구성되어 있는 경우 어떻게 연결하는지 보여줍니다. `LOCAL_MCP_COMMAND`가 설정되어 있지 않으면 우아하게 건너뛰므로 노트북이 끝까지 실행됩니다.

보안 참고: 로컬 MCP 서버는 사용자의 권한으로 실행됩니다. 프로젝트 디렉터리에 범위를 제한하고 그 출력을 검증한 후에 처리하세요.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## 요약

여러분은 완전히 로컬에서 실행되는 엔지니어링 어시스턴트를 구축했습니다:

- <strong>Foundry Local</strong>는 OpenAI 호환 엔드포인트 뒤에 **Qwen** 모델을 제공하여 에이전트 코드가 클라우드 수업과 일치하도록 했습니다.
- <strong>Sandboxed tools</strong>는 프로젝트 디렉터리를 벗어나지 않고 에이전트에게 파일 액세스 및 코드 분석 기능을 제공했습니다.
- <strong>Chroma</strong>는 문서에 대한 <strong>로컬 RAG</strong>를 제공했습니다.
- <strong>Local MCP</strong>는 MCP 생태계를 오프라인에서 재사용하는 방법을 보여주었습니다.

어느 순간에도 클라우드 추론이 사용되지 않았습니다.

### 도전 과제

로컬 에이전트를 다음과 같이 확장하세요:

1. **여러 MCP 서버 작업** — 파일 시스템 서버와 git 서버를 연결하고 에이전트가 둘 중 하나를 선택하도록 합니다.
2. **로컬 메모리 사용** — 간단한 대화 기록을 디스크에 저장하여 어시스턴트가 노트북 재시작 시 이전 대화를 기억하게 합니다.
3. **로컬 다중 에이전트 오케스트레이션 지원** — 두 번째 로컬 에이전트(예: 리뷰어)를 추가하고 두 에이전트가 작업을 협력하도록 합니다.

다음 수업에서는 배포된 AI 에이전트를 보호하는 방법을 배우게 됩니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
